# Mini Project 1 — Analysis Notebook

**Your name:** Gena Lee
**Dataset:**  last.fm_global_top_artists.csv
**Date:**  5/6/2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [1]:
# Setup — run this cell first
# Put your key in .env in this same folder: LASTFM_API_KEY=... (no spaces around =)
# Or set the env var in your shell/IDE instead.
#
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** My data comes from the Last.fm API (last.fm/api), a free public API that aggregates real listening behavior from millions of users across streaming platforms like Spotify and Apple Music. Last.fm tracks individual song plays and exposes aggregated statistics through its API.

I will access the data using Python's requests library, pulling JSON responses directly into pandas DataFrames. No authentication beyond a free API key is required for the public read-only endpoints I plan to use.

**Why this dataset:** *I chose the Last.fm API because of my interests in music and how it relates on a global scale. I think it would be super interesting to see how music is perceived and listened to across countries. I would also be open to focusing on specific countries (e.g. US, Korea, Brazil) if I have time.*

**Three analytical questions:**

1. *Which artist has the most devoted fanbase?*
2. *Which genres have the most listeners?*
3. *Among the top 100 global artists, which genres have the highest play-to-listener ratio?*


**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [ ]:
# Global top artists from Last.fm (reads .env in this folder for LASTFM_API_KEY)
import json
import os
import urllib.parse
import urllib.request
from pathlib import Path

_env = Path.cwd() / ".env"
if _env.is_file():
    for _line in _env.read_text(encoding="utf-8").splitlines():
        _line = _line.strip()
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _, _v = _line.partition("=")
            _k, _v = _k.strip(), _v.strip().strip('"').strip("'")
            if _k:
                os.environ.setdefault(_k, _v)

api_key = (os.environ.get("LASTFM_API_KEY") or "").strip()
if not api_key:
    raise ValueError("Add LASTFM_API_KEY to .env in this folder (or set it in your environment).")

#I put in a TOP_N to limit the artists to 50
TOP_N = 50
params = {
    "method": "chart.gettopartists",
    "api_key": api_key,
    "format": "json",
    "limit": str(TOP_N),
}
url = "https://ws.audioscrobbler.com/2.0/?" + urllib.parse.urlencode(params)
with urllib.request.urlopen(url, timeout=30) as resp:
    payload = json.load(resp)

if payload.get("error") is not None:
    raise RuntimeError(payload.get("message", str(payload)))

artists = payload.get("artists", {}).get("artist", [])
if isinstance(artists, dict):
    artists = [artists]


def _to_int(x, default=0):
    try:
        return int(x)
    except (TypeError, ValueError):
        return default

# Here I am building what I want the dataframe or rows to look like
rows = []
for i, item in enumerate(artists, start=1):
    attrs = item.get("@attr", {})
    rows.append(
        {
            "rank": _to_int(attrs.get("rank"), default=i),
            "name": item.get("name", ""),
            "playcount": _to_int(item.get("playcount")),
            "listeners": _to_int(item.get("listeners")),
            "mbid": item.get("mbid") or "",
            "url": item.get("url", ""),
        }
    )
#I wanted to rank the artists by playcount, so I created a new column called rank_by_playcount
df = pd.DataFrame(rows)
df = df.sort_values("playcount", ascending=False).reset_index(drop=True)
df["rank_by_playcount"] = df.index + 1

print(df.shape)
df.head()

(50, 7)


,rank,name,playcount,listeners,mbid,url,rank_by_playcount
0,49,BTS,3864260712,2462420,0d79fe8e-ba27-4859-bb8c-2f255f346853,https://www.last.fm/music/BTS,1
1,12,Taylor Swift,3688324282,5954626,20244d07-534f-4eff-b4d4-930878889970,https://www.last.fm/music/Taylor+Swift,2
2,16,Lana Del Rey,1588131650,5274637,b7539c32-53e7-4908-bda3-81449c367da6,https://www.last.fm/music/Lana+Del+Rey,3
3,2,Kanye West,1549466939,7968237,,https://www.last.fm/music/Kanye+West,4
4,11,Radiohead,1379940956,8254057,a74b1b7f-71a5-4011-9441-d0b5e4122711,https://www.last.fm/music/Radiohead,5


In [5]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   rank               50 non-null     int64
 1   name               50 non-null     str  
 2   playcount          50 non-null     int64
 3   listeners          50 non-null     int64
 4   mbid               50 non-null     str  
 5   url                50 non-null     str  
 6   rank_by_playcount  50 non-null     int64
dtypes: int64(4), str(3)
memory usage: 2.9 KB


In [6]:
# Summary statistics for numeric columns
df.describe()

,rank,playcount,listeners,rank_by_playcount
count,50.00000,5.000000e+01,5.000000e+01,50.00000
mean,25.50000,7.564742e+08,4.695715e+06,25.50000
std,14.57738,7.150820e+08,1.895152e+06,14.57738
min,1.00000,8.979104e+07,1.343457e+06,1.00000
25%,13.25000,3.656697e+08,3.272107e+06,13.25000
50%,25.50000,5.838484e+08,4.311340e+06,25.50000
75%,37.75000,8.273128e+08,5.880705e+06,37.75000
max,50.00000,3.864261e+09,9.124928e+06,50.00000


**Your data profile notes:**  
*I found that the original ranking of the top artists weren't ranked by playcount or listeners. So, I had to write a new code to rank the artists by playcount, which gave me a more accurate ranking.*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** *Which artist has the most loyal fanbase?*

In [28]:
# Your analysis for Question 1


**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Question 2:** *(paste your second research question here)*

In [29]:
# Your analysis for Question 2


**Interpretation:**  
*(What does this result tell you?)*

**Question 3:** *(paste your third research question here)*

In [30]:
# Your analysis for Question 3


**Interpretation:**  
*(What does this result tell you?)*

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [31]:
# Your visualization
# Example structure — replace with your actual columns and finding

# fig = px.bar(
#     df,
#     x="your_category_column",
#     y="your_value_column",
#     title="Your finding stated as a claim",
#     labels={"your_category_column": "Readable label", "your_value_column": "Readable label"}
# )
# fig.show()


**Chart rationale:**  
*(Why this chart type? What should the reader take away?)*

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.